In [ ]:
import os
import numpy as np
import torch
import pandas as pd
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from torch.utils.data import DataLoader
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, f1_score
from tqdm.auto import tqdm

from script.phobert_dataset import PhoBERTDataset

## 1. Cấu hình

In [ ]:
DATA_FILE    = "health_classification_balanced.csv"
MODEL_NAME   = "vinai/phobert-base"
SAVE_DIR     = "phobert_health_model"

MAX_LEN      = 256
BATCH_SIZE   = 16
EPOCHS       = 5
LR           = 2e-5
WARMUP_RATIO = 0.1   # 10% đầu dùng warmup learning rate
WEIGHT_DECAY = 0.01
PATIENCE     = 2     # Early stopping: dừng nếu val F1 không cải thiện sau 2 epoch
SEED         = 42

torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## 2. Load dataset

In [ ]:
df     = pd.read_csv(DATA_FILE)
texts  = df["text"].astype(str).tolist()
labels = df["is_health"].tolist()

print(f"Dataset size: {len(df)}")
print(df["is_health"].value_counts())

## 3. Tokenizer · Train/Val split · DataLoader

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

X_train, X_val, y_train, y_val = train_test_split(
    texts, labels,
    test_size=0.2,
    stratify=labels,
    random_state=SEED,
)
print(f"Train: {len(X_train)} | Val: {len(X_val)}")

train_loader = DataLoader(
    PhoBERTDataset(X_train, y_train, tokenizer, MAX_LEN),
    batch_size=BATCH_SIZE, shuffle=True,
)
val_loader = DataLoader(
    PhoBERTDataset(X_val, y_val, tokenizer, MAX_LEN),
    batch_size=BATCH_SIZE,
)

## 4. Model

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    hidden_dropout_prob=0.2,
    attention_probs_dropout_prob=0.2,
)
model.to(device)

## 5. Optimizer · Scheduler

In [ ]:
# Không áp weight decay lên bias và LayerNorm (best practice cho Transformer)
no_decay = ["bias", "LayerNorm.weight"]
param_groups = [
    {
        "params": [p for n, p in model.named_parameters()
                   if not any(nd in n for nd in no_decay)],
        "weight_decay": WEIGHT_DECAY,
    },
    {
        "params": [p for n, p in model.named_parameters()
                   if any(nd in n for nd in no_decay)],
        "weight_decay": 0.0,
    },
]

optimizer    = AdamW(param_groups, lr=LR)
total_steps  = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

# Linear warmup → linear decay
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)
print(f"Total steps: {total_steps} | Warmup steps: {warmup_steps}")

## 6. Training loop

In [ ]:
os.makedirs(SAVE_DIR, exist_ok=True)
best_val_f1    = 0.0
patience_count = 0

for epoch in range(EPOCHS):
    print(f"\n{'='*50}\nEpoch {epoch + 1}/{EPOCHS}\n{'='*50}")

    # ── Train ────────────────────────────────────────────
    model.train()
    total_loss = 0.0

    for batch in tqdm(train_loader, desc="Training"):
        optimizer.zero_grad()

        outputs = model(
            input_ids      = batch["input_ids"].to(device),
            attention_mask = batch["attention_mask"].to(device),
            labels         = batch["labels"].to(device),
        )
        outputs.loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        total_loss += outputs.loss.item()

    print(f"Train Loss: {total_loss / len(train_loader):.4f} | LR: {scheduler.get_last_lr()[0]:.2e}")

    # ── Validation ───────────────────────────────────────
    model.eval()
    preds, trues = [], []

    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Validating"):
            outputs = model(
                input_ids      = batch["input_ids"].to(device),
                attention_mask = batch["attention_mask"].to(device),
            )
            preds.extend(outputs.logits.argmax(dim=1).cpu().numpy())
            trues.extend(batch["labels"].numpy())

    val_f1 = f1_score(trues, preds, average="weighted")
    print(f"Val Accuracy: {accuracy_score(trues, preds):.4f} | Val F1: {val_f1:.4f}")
    print(classification_report(trues, preds, digits=4, target_names=["Non-health", "Health"]))

    if val_f1 > best_val_f1:
        best_val_f1    = val_f1
        patience_count = 0
        model.save_pretrained(SAVE_DIR)
        tokenizer.save_pretrained(SAVE_DIR)
        print(f"✅ Saved best model (F1={best_val_f1:.4f})")
    else:
        patience_count += 1
        print(f"⚠️  No improvement ({patience_count}/{PATIENCE})")
        if patience_count >= PATIENCE:
            print("🛑 Early stopping triggered!")
            break

print(f"\nBest Val F1: {best_val_f1:.4f}")

## 7. Quick test (best model)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(SAVE_DIR).to(device)
model.eval()

test_texts = [
    "Bộ Y tế khuyến cáo người dân phòng chống sốt xuất huyết",
    "Giá vàng hôm nay tăng mạnh vượt 100 triệu đồng/lượng",
    "Phát hiện thuốc điều trị ung thư gan giai đoạn cuối hiệu quả cao",
]

print("--- Kết quả test ---")
for text in test_texts:
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LEN,
        padding=True,
    ).to(device)

    with torch.no_grad():
        logits = model(**inputs).logits
        pred   = logits.argmax(dim=1).item()
        conf   = torch.softmax(logits, dim=1).cpu().numpy()[0][pred]

    label = "✅ Y tế" if pred == 1 else "❌ Không phải y tế"
    print(f"\nText : {text}")
    print(f"Nhãn : {label} | Confidence: {conf * 100:.1f}%")